# AbdomenAtlas ↔ AdrenalGold overlap — shards 06–19  (no Google Drive, GPU optional)

Finds gold-standard scans (AMOS/BTCV/FLARE/RAOS) physically present in AbdomenAtlas 1.1 Mini
by replicating AbdomenAtlas's standardization (clip → largest 26-connected-component body crop →
reorient) and matching by byte-md5 (exact) / shape+32³-corr against precomputed **gold** descriptors.

**Setup (no Drive needed):**
1. Runtime → A100/L4 is faster (more vCPUs) but **CPU-only works**; no GPU required.
2. HF token: either add a Colab secret `HF_TOKEN` (🔑 sidebar), set env var `HF_TOKEN`, or you'll be prompted.
3. Gold cache `atlas_std_cache.pkl` (170 MB): you'll be prompted to **upload** it (it's your local
   `AdrenalGold/.atlas_std_cache.pkl`). No Drive mount.

**Output:** `atlas_overlap_0619.csv` in the session working dir; auto-downloaded at the end (and the
summary cell can re-download it). Re-running resumes from shards already in the CSV (within a session).
Concatenate with your local `atlas_overlap.csv` (shards 01–05) for the full 19-shard table.

In [ ]:
import os, subprocess, multiprocessing as mp
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
subprocess.run("pip -q install nibabel scipy scikit-image huggingface_hub hf_transfer requests".split())
try:
    GPU = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
except Exception:
    GPU = False            # nvidia-smi absent on CPU-only -> FileNotFoundError; handled here
print("GPU present:", GPU, "| CPU cores:", mp.cpu_count(), "(GPU not required)")

In [ ]:
import os, getpass
try:
    import google.colab            # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False
WORK = "/content" if IN_COLAB else os.getcwd()
os.makedirs(WORK, exist_ok=True)

# HF token: env var -> Colab secret -> prompt
TOKEN = os.environ.get("HF_TOKEN", "")
if not TOKEN and IN_COLAB:
    try:
        from google.colab import userdata
        TOKEN = userdata.get("HF_TOKEN") or ""
    except Exception:
        TOKEN = ""
if not TOKEN:
    TOKEN = getpass.getpass("HF token: ").strip()

# gold cache (NO Google Drive) - accept dot or non-dot name; upload in Colab if missing
CAND = [os.path.join(WORK, "atlas_std_cache.pkl"), os.path.join(WORK, ".atlas_std_cache.pkl")]
CACHE = next((c for c in CAND if os.path.exists(c)), CAND[0])
if not os.path.exists(CACHE):
    if IN_COLAB:
        from google.colab import files
        print("Upload your local AdrenalGold/.atlas_std_cache.pkl (170 MB):")
        up = files.upload()
        os.replace(list(up)[0], CACHE)
    else:
        raise SystemExit("Place atlas_std_cache.pkl in: " + WORK)
CSV = os.path.join(WORK, "atlas_overlap_0619.csv")
print("IN_COLAB:", IN_COLAB, "| cache:", round(os.path.getsize(CACHE)/1e6, 1), "MB | output:", CSV)

In [ ]:
import os, hashlib, pickle
import numpy as np, nibabel as nib
from nibabel.orientations import io_orientation, apply_orientation
from scipy import ndimage as ndi
from scipy.ndimage import zoom
STRUCT = np.ones((3, 3, 3), bool)            # 26-conn == AbdomenAtlas skimage default
REPO = "BodyMaps/_AbdomenAtlas1.1Mini"

def standardize(path):
    # MUST match local .atlas_match3.py exactly so md5s line up with the gold cache
    try:
        img = nib.load(path); data = np.array(img.dataobj)
        if data.ndim != 3: data = data.squeeze()
        data[data > 1000] = 1000; data[data < -1000] = -1000
        cond = (data > -100) & (data < 100)
        lab, n = ndi.label(cond, structure=STRUCT)
        if n == 0: return (path, None, None, None)
        counts = np.bincount(lab.ravel()); counts[0] = 0
        largest = int(counts.argmax())
        sl = ndi.find_objects(lab)[largest - 1]
        crop = data[sl]
        ras = apply_orientation(crop, io_orientation(img.affine)).astype(np.int16)
        shp = tuple(int(s) for s in ras.shape)
        md5 = hashlib.md5(np.ascontiguousarray(ras)).hexdigest()
        w = np.clip(ras.astype(np.float32), -200, 300)
        d = zoom(w, tuple(32.0 / s for s in w.shape), order=1)[:32, :32, :32]
        if d.shape != (32, 32, 32):
            d = np.pad(d, [(0, 32 - d.shape[i]) for i in range(3)])
        v = d.ravel().astype(np.float32); v -= v.mean(); s = v.std()
        if s > 0: v /= s
        return (path, shp, md5, v)
    except Exception:
        return (path, None, None, None)

def load_gold(cache_path):
    cache = pickle.load(open(cache_path, "rb"))
    G = [r for r in cache if r[1] != "ATLAS" and r[2] is not None]   # (path, ds, shape, md5, v)
    g_md5 = {}
    for path, ds, shp, md5, v in G:
        g_md5.setdefault(md5, []).append((ds, path, shp))
    GV = np.stack([r[4] for r in G])
    gmeta = [(r[1], r[0], r[2]) for r in G]
    return g_md5, GV, gmeta, GV.shape[1]

def gname(ds, gp):
    return os.path.basename(os.path.dirname(gp)) if ds == "RAOS" else os.path.basename(gp)

def match_rows(stag, results, g_md5, GV, gmeta, N):
    rows = []
    for path, shp, md5, v in results:
        if shp is None: continue
        bd = path.split("/")[-2]
        if md5 in g_md5:
            for ds, gp, gshp in g_md5[md5]:
                rows.append((stag, ds, gname(ds, gp), bd, "exact", "1.0000"))
            continue
        corr = (GV @ v) / N; j = int(np.argmax(corr)); bestc = float(corr[j])
        ds, gp, gshp = gmeta[j]
        if bestc > 0.97 and gshp == shp:
            rows.append((stag, ds, gname(ds, gp), bd, "shape+corr", f"{bestc:.4f}"))
        elif bestc > 0.99:
            rows.append((stag, ds, gname(ds, gp), bd, "corr-only", f"{bestc:.4f}"))
    return rows


In [ ]:
import glob, time, csv, tarfile, shutil
from collections import Counter
from huggingface_hub import hf_hub_download
SHARDS = {"06":"00002501_00003000","07":"00003001_00003500","08":"00003501_00004000",
          "09":"00004001_00004500","10":"00004501_00005000","11":"00005001_00005500",
          "12":"00005501_00006000","13":"00006001_00006500","14":"00006501_00007000",
          "15":"00007001_00007500","16":"00007501_00008000","17":"00008001_00008500",
          "18":"00008501_00009000","19":"00009001_00009262"}
HEADER = ["shard","gold_dataset","gold_scan","atlas_bdmap_id","match_type","corr32"]
g_md5, GV, gmeta, N = load_gold(CACHE)
NW = max(2, mp.cpu_count())
DL = os.path.join(WORK, "dl")

def read_rows():
    if not os.path.exists(CSV): return []
    return [x for x in list(csv.reader(open(CSV)))[1:] if x]
def write_shard(stag, rows):
    keep = [x for x in read_rows() if x[0] != stag]
    with open(CSV, "w", newline="") as f:
        w = csv.writer(f); w.writerow(HEADER); w.writerows(keep); w.writerows(sorted(rows))

done = set(x[0] for x in read_rows())
for stag, rng in SHARDS.items():
    tag = "shard" + stag
    if tag in done:
        print("skip", tag, "(already in CSV)"); continue
    t0 = time.time(); tb = "AbdomenAtlas1.1Mini_BDMAP_%s.tar.gz" % rng
    print("[%s] downloading %s" % (stag, tb), flush=True)
    tp = hf_hub_download(REPO, tb, repo_type="dataset", token=TOKEN, local_dir=DL)
    print("[%s] %.1f GB in %ds; extracting ct" % (stag, os.path.getsize(tp)/1e9, time.time()-t0), flush=True)
    cdir = os.path.join(WORK, "ct_" + stag); os.makedirs(cdir, exist_ok=True)
    with tarfile.open(tp, "r:gz") as tf:
        for m in tf:
            if m.name.endswith("ct.nii.gz"): tf.extract(m, cdir)
    cts = sorted(glob.glob(cdir + "/**/ct.nii.gz", recursive=True))
    print("[%s] %d ct; standardizing on %d cores" % (stag, len(cts), NW), flush=True)
    with mp.Pool(NW) as pool:
        res = pool.map(standardize, cts, chunksize=4)
    rows = match_rows(tag, res, g_md5, GV, gmeta, N)
    write_shard(tag, rows)
    print("[%s] DONE %ds | %d matches %s" % (stag, time.time()-t0, len(rows), dict(Counter(r[1] for r in rows))), flush=True)
    os.remove(tp); shutil.rmtree(cdir, ignore_errors=True); shutil.rmtree(DL, ignore_errors=True)
print("=== ALL SHARDS 06-19 DONE ->", CSV, "===")

In [ ]:
import csv
from collections import Counter
rows = [x for x in list(csv.reader(open(CSV)))[1:] if x]
print("total matches 06-19:", len(rows))
print("by shard:", dict(sorted(Counter(r[0] for r in rows).items())))
print("by dataset:", dict(Counter(r[1] for r in rows)))
print("by type:", dict(Counter(r[4] for r in rows)))
try:
    from google.colab import files; files.download(CSV)
except Exception:
    print("CSV saved at:", CSV)